# درس 9: سری زمانی: ایستایی و خودهمبستگی

متن کامل درس: [https://quantacademy.ir/courses/quant-math/time-series-stationarity-autocorrelation/](https://quantacademy.ir/courses/quant-math/time-series-stationarity-autocorrelation/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. ایستایی یعنی چه؟

In [ ]:
import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
ret = np.log(close).diff().dropna()

half = len(ret) // 2
print(f"price  mean: first half {close.iloc[:half].mean():,.0f}  second half {close.iloc[half:].mean():,.0f}")
print(f"return mean: first half {ret.iloc[:half].mean():.5f}  second half {ret.iloc[half:].mean():.5f}")
print(f"return std : first half {ret.iloc[:half].std():.4f}  second half {ret.iloc[half:].std():.4f}")

## ۲. تابع خودهمبستگی (ACF)

In [ ]:
def acf(x, k):
    x = np.asarray(x, dtype=float) - np.mean(x)
    return float(np.sum(x[k:] * x[:-k]) / np.sum(x * x))

lags = [1, 5, 20]
print("price  ", [round(acf(close.values, k), 3) for k in lags])
print("return ", [round(acf(ret.values, k), 3) for k in lags])
print("band   ", round(2 / np.sqrt(len(ret)), 3))

## ۳. نوسان خوشه‌ای

In [ ]:
lags = [1, 2, 5, 10, 20]
print("lag     ", lags)
print("return  ", [round(acf(ret.values, k), 3) for k in lags])
print("|return|", [round(acf(np.abs(ret.values), k), 3) for k in lags])
print("return^2", [round(acf(ret.values ** 2, k), 3) for k in lags])

## ۴. مدل AR(1) و نیمه‌عمر

In [ ]:
def simulate_ar1(phi, n, seed):
    rng = np.random.default_rng(seed)
    e = rng.normal(0, 1, n)
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = phi * x[t - 1] + e[t]
    return x

phi, n = 0.9, 2000
x = simulate_ar1(phi, n, seed=4)
print("lag    ", [1, 2, 5, 10])
print("sample ", [round(acf(x, k), 3) for k in (1, 2, 5, 10)])
print("theory ", [round(phi ** k, 3) for k in (1, 2, 5, 10)])

In [ ]:
y, z = x[1:], x[:-1]
phi_hat = np.sum(z * y) / np.sum(z * z)
resid = y - phi_hat * z
se = np.sqrt(resid.var(ddof=1) / np.sum(z * z))
half_life = lambda p: np.log(0.5) / np.log(p)
print(f"phi_hat {phi_hat:.4f}  se {se:.4f}")
print(f"half-life: true {half_life(phi):.2f}  estimated {half_life(phi_hat):.2f}")
print(f"half-life range (phi_hat +- 2 se): {half_life(phi_hat - 2 * se):.2f} .. {half_life(phi_hat + 2 * se):.2f}")

## ۵. سوگیری در نمونه کوچک

In [ ]:
rng = np.random.default_rng(5)
reps, m, phi_true = 5000, 100, 0.95
e = rng.normal(0, 1, (reps, m))
paths = np.zeros((reps, m))
for t in range(1, m):
    paths[:, t] = phi_true * paths[:, t - 1] + e[:, t]
zz, yy = paths[:, :-1], paths[:, 1:]
phis = (zz * yy).sum(axis=1) / (zz * zz).sum(axis=1)
print(f"true phi {phi_true}  mean of estimates {phis.mean():.3f}  5%..95%: {np.percentile(phis, 5):.3f} .. {np.percentile(phis, 95):.3f}")

## ۶. آزمون Dickey-Fuller (نسخه ساده)

In [ ]:
def dickey_fuller(x):
    """Simplified Dickey-Fuller: regress the change on the previous level, no constant, no extra lags."""
    dx, z = np.diff(x), x[:-1]
    beta = np.sum(z * dx) / np.sum(z * z)
    r = dx - beta * z
    se = np.sqrt(r.var(ddof=1) / np.sum(z * z))
    return beta / se

rw = np.cumsum(np.random.default_rng(6).normal(0, 1, 2000))
log_price = np.log(close.values)
print(f"AR(1) phi=0.9      t = {dickey_fuller(x):8.2f}")
print(f"random walk        t = {dickey_fuller(rw):8.2f}")
print(f"BTC log price      t = {dickey_fuller(log_price - log_price.mean()):8.2f}")
print(f"BTC log return     t = {dickey_fuller(ret.values):8.2f}")
print("critical value (5%) about -1.94")